# Course 6: Geometric Deep Learning and Graphs
## Session 2: Message passing

**Goal:** train a two-layer GCN for node classification and compare it with a model that ignores the graph.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on small built-in or synthetic data. Cells labelled NOT RUN need a library, a dataset download or a GPU; run them on Kaggle or Colab.

### 1. A citation-like graph
Cora needs a download, so the runnable part uses a synthetic graph with the same character: papers in the same topic cite each other more often, and the features are noisy.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

def sbm(sizes, p_in, p_out, rng):
    """Stochastic block model: dense inside a block, sparse between blocks."""
    labels = np.repeat(np.arange(len(sizes)), sizes)
    n = len(labels)
    P = np.where(labels[:, None] == labels[None, :], p_in, p_out)
    A = np.triu(rng.random((n, n)) < P, 1)
    A = (A | A.T).astype(np.float32)
    return A, labels

def norm_adj(A):
    """GCN normalization: D^-1/2 (A + I) D^-1/2."""
    A = A + np.eye(len(A), dtype=A.dtype)
    d = A.sum(1)
    return A / np.sqrt(np.outer(d, d))

class GCN(nn.Module):
    def __init__(self, d_in, d_hid, d_out):
        super().__init__()
        self.l1, self.l2 = nn.Linear(d_in, d_hid), nn.Linear(d_hid, d_out)
    def forward(self, X, A_hat):
        H = F.relu(A_hat @ self.l1(X))
        return A_hat @ self.l2(H)

In [ ]:
rng = np.random.default_rng(0); torch.manual_seed(0)
A, y = sbm([100] * 4, p_in=0.06, p_out=0.008, rng=rng)
n, k = len(y), 4
X = np.eye(k)[y] @ rng.normal(size=(k, 32)) * 0.35 + rng.normal(size=(n, 32))   # weak class signal + noise
idx = rng.permutation(n); train, test = idx[:40], idx[200:]                      # only 40 labelled nodes
X, y_t, A_hat = torch.tensor(X, dtype=torch.float32), torch.tensor(y), torch.tensor(norm_adj(A))
print("nodes", n, "edges", int(A.sum() / 2), "labelled", len(train))

### 2. Train a GCN and an MLP on the same 40 labels

In [ ]:
def fit(model, use_graph, epochs=200):
    opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    I = torch.eye(n)
    for _ in range(epochs):
        out = model(X, A_hat if use_graph else I)
        loss = F.cross_entropy(out[train], y_t[train])
        opt.zero_grad(); loss.backward(); opt.step()
    pred = model(X, A_hat if use_graph else I).argmax(1)
    return (pred[test] == y_t[test]).float().mean().item()

acc_mlp = fit(GCN(32, 32, k), use_graph=False)     # identity adjacency = no message passing
acc_gcn = fit(GCN(32, 32, k), use_graph=True)
print(f"MLP (no graph) test accuracy: {acc_mlp:.3f}")
print(f"GCN            test accuracy: {acc_gcn:.3f}")
assert acc_gcn > acc_mlp

### 3. The same recipe, three aggregators
GCN, GraphSAGE and GAT differ only in how a node combines its neighbours' messages.

In [ ]:
A_t = torch.tensor(A)
H = torch.randn(n, 8)
gcn_agg  = A_hat @ H                                              # degree-normalized sum, self included
sage_agg = torch.cat([H, (A_t @ H) / A_t.sum(1, keepdim=True).clamp(min=1)], 1)   # self, then mean of neighbours
a = torch.randn(16)                                               # GAT: attention score from the pair (i, j)
e = F.leaky_relu((H @ a[:8])[:, None] + (H @ a[8:])[None, :], 0.2)
alpha = torch.softmax(e.masked_fill(A_t + torch.eye(n) == 0, float("-inf")), dim=1)
gat_agg = alpha @ H
print(gcn_agg.shape, sage_agg.shape, gat_agg.shape, "| attention rows sum to", alpha.sum(1)[:3])

### 4. Cora with PyTorch Geometric (NOT RUN here)

In [ ]:
# NOT RUN: needs `pip install torch_geometric` and a dataset download.
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv
ds = Planetoid("data", "Cora"); g = ds[0]

class Net(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.c1, self.c2 = GCNConv(ds.num_features, 16), GCNConv(16, ds.num_classes)
    def forward(self, x, ei):
        x = F.dropout(F.relu(self.c1(x, ei)), 0.5, self.training)
        return self.c2(x, ei)

net = Net(); opt = torch.optim.Adam(net.parameters(), lr=0.01, weight_decay=5e-4)
for epoch in range(200):
    net.train(); opt.zero_grad()
    F.cross_entropy(net(g.x, g.edge_index)[g.train_mask], g.y[g.train_mask]).backward(); opt.step()
net.eval(); pred = net(g.x, g.edge_index).argmax(1)
print("test accuracy", (pred[g.test_mask] == g.y[g.test_mask]).float().mean().item())   # the GCN paper reports 81.5%

### TODO
1. Run the Cora cell and report mean and standard deviation over 5 seeds.
2. Swap `GCNConv` for `SAGEConv` and `GATConv`. Which is best, and is the gap larger than the seed noise?
3. In the synthetic graph, raise `p_out` until the GCN stops beating the MLP. Why does that happen?